In [ ]:
USE DATABASE FINAL_PROJ_DB;
USE SCHEMA RAW_DATA;
CREATE OR REPLACE NETWORK RULE dbt_packages_network_rule
  MODE = EGRESS
  TYPE = HOST_PORT
  VALUE_LIST = (
    'hub.getdbt.com',      -- The dbt Package hub
    'codeload.github.com', -- Packages hosted on GitHub
    'github.com'           -- GitHub repositories
  );


In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION dbt_external_access_integration
  ALLOWED_NETWORK_RULES = (dbt_packages_network_rule)
  ENABLED = TRUE;

In [ ]:
%%sql -r dataframe_2
GRANT USAGE ON INTEGRATION dbt_external_access_integration TO ROLE project_member;
GRANT CREATE DBT PROJECT ON SCHEMA "FINAL_PROJ_DB"."RAW_DATA" TO ROLE project_member;

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE STREAM final_proj_db.raw_data.raw_storm_events_stream
  ON TABLE final_proj_db.raw_data.raw_storm_events
  APPEND_ONLY = TRUE;  -- Snowpipe only inserts, no updates/deletes

In [ ]:
%%sql -r dataframe_7
CREATE OR REPLACE TRANSIENT TABLE final_proj_db.raw_data.stream_trigger_log (
    fired_at TIMESTAMP, 
    row_count NUMBER
);

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE TASK final_proj_db.raw_data.ack_storm_events_trigger
  WAREHOUSE = final_project_wh
  WHEN SYSTEM$STREAM_HAS_DATA('final_proj_db.raw_data.raw_storm_events_stream')
AS
  INSERT INTO final_proj_db.raw_data.stream_trigger_log (fired_at, row_count)
SELECT CURRENT_TIMESTAMP(), COUNT(*)
FROM final_proj_db.raw_data.raw_storm_events_stream;


In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE TASK final_proj_db.raw_data.run_dbt_build
  WAREHOUSE = final_project_wh
  AFTER final_proj_db.raw_data.ack_storm_events_trigger
AS
  EXECUTE DBT PROJECT final_proj_db.raw_data.SEVERE_WEATHER_RISK_DBT
    ARGS = 'build --select intermediate_staging_table+';


In [ ]:
%%sql -r dataframe_15
CREATE OR REPLACE TASK final_proj_db.raw_data.run_dbt_test
  WAREHOUSE = final_project_wh
  AFTER final_proj_db.raw_data.run_dbt_build
AS
  EXECUTE DBT PROJECT final_proj_db.raw_data.SEVERE_WEATHER_RISK_DBT
    ARGS = 'test';


In [ ]:
%%sql -r dataframe_6
ALTER TASK final_proj_db.raw_data.run_dbt_test RESUME;
ALTER TASK final_proj_db.raw_data.run_dbt_build RESUME;
ALTER TASK final_proj_db.raw_data.ack_storm_events_trigger RESUME;

In [ ]:
%%sql -r dataframe_10
ALTER TASK final_proj_db.raw_data.ack_storm_events_trigger SUSPEND;
ALTER TASK final_proj_db.raw_data.run_dbt_build SUSPEND;
ALTER TASK final_proj_db.raw_data.run_dbt_test SUSPEND;

In [ ]:
%%sql -r dataframe_18
select max(_loaded_at) from final_proj_db.raw_data.raw_storm_events;

In [ ]:
%%sql -r dataframe_8
delete FROM final_proj_db.raw_data.raw_storm_events
WHERE _loaded_at >= '2026-08-21 08:50:16';

In [ ]:
delete FROM final_proj_db.silver.INTERMEDIATE_STAGING_TABLE
WHERE _loaded_at >= '2026-08-21 08:00:16';

In [ ]:
delete FROM final_proj_db.silver.silver_storm_events_quarantine
WHERE _loaded_at >= '2026-08-21 08:00:16';

In [ ]:
delete FROM final_proj_db.silver.silver_storm_events
WHERE _loaded_at >= '2026-08-21 08:00:00';

In [ ]:
%%sql -r dataframe_14
 delete FROM final_proj_db.gold.fct_storm_events
WHERE _loaded_at >= '2026-08-21 08:00:16';